# ISOM5240 Model Comparison — Pipeline 1 & Pipeline 3

**Pipeline 1:** Product image → Shelf-life category (short / medium / non-perishable)  
**Pipeline 3:** Product image → Auto-generated text description  

Both pipelines use pre-trained models. This notebook compares 3 candidates for each and selects the best.

**Datasets used for Pipeline 1 testing:**
- Grocery Store Dataset (GitHub) → short_shelf + medium_shelf
- Kaggle Household Products → non_perishable

## Step 1: Install dependencies

In [ ]:
!pip install transformers pillow -q

## Step 2: GPU check

In [ ]:
import torch
import os
import time
import numpy as np

if torch.cuda.is_available():
    DEVICE = 0
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    DEVICE = -1
    print("No GPU, using CPU (fine — no training needed)")

## Step 3: Setup Kaggle credentials

In [ ]:
from google.colab import userdata

os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')
print("Kaggle credentials set")

## Step 4: Download both datasets

1. Grocery Store Dataset (GitHub) → fruits, vegetables, dairy
2. Kaggle Household Products → tissue, cleaning supplies, etc.

In [ ]:
# Dataset 1: Grocery Store Dataset
!git clone https://github.com/marcusklasson/GroceryStoreDataset.git
print("Grocery Store Dataset downloaded")

# Dataset 2: Household Products
!kaggle datasets download -d taru149/householdproducts
!unzip -q householdproducts.zip -d household/
print("Household Products downloaded")

# Check structure
print("\nGrocery Store test folders:")
grocery_test = "GroceryStoreDataset/dataset/test"
if os.path.exists(grocery_test):
    print(os.listdir(grocery_test))

print("\nHousehold folders:")
!find household/ -type d | head -20

## Step 5: Build combined 3-class test set

- short_shelf (0): fruits, vegetables from Grocery Store Dataset
- medium_shelf (1): dairy, juice from Grocery Store Dataset
- non_perishable (2): household products from Kaggle

In [ ]:
from PIL import Image
import glob

# === Shelf-life keywords for Grocery Store Dataset folders ===
SHORT_KEYWORDS = [
    "apple", "avocado", "banana", "kiwi", "lemon", "lime", "mango",
    "melon", "nectarine", "orange", "papaya", "passion", "peach",
    "pear", "pineapple", "plum", "pomegranate", "grapefruit",
    "satsuma", "watermelon", "asparagus", "aubergine", "cabbage",
    "carrot", "cucumber", "garlic", "ginger", "leek", "mushroom",
    "onion", "pepper", "potato", "red-beet", "tomato", "zucchini"
]

MEDIUM_KEYWORDS = [
    "juice", "milk", "oat", "sour-cream", "sour-milk",
    "soy", "yoghurt", "cream"
]

LABEL_NAMES_P1 = ["short_shelf", "medium_shelf", "non_perishable"]

# Collect from Grocery Store Dataset
test_images = []
test_labels = []

grocery_test = "GroceryStoreDataset/dataset/test"
if os.path.exists(grocery_test):
    for class_dir in sorted(os.listdir(grocery_test)):
        class_path = os.path.join(grocery_test, class_dir)
        if not os.path.isdir(class_path):
            continue

        name = class_dir.lower()
        if any(kw in name for kw in SHORT_KEYWORDS):
            label = 0
        elif any(kw in name for kw in MEDIUM_KEYWORDS):
            label = 1
        else:
            continue

        for img_file in os.listdir(class_path):
            if img_file.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".gif", ".webp")):
                test_images.append(os.path.join(class_path, img_file))
                test_labels.append(label)

# Collect from Household Products (all = non_perishable)
household_paths = (glob.glob("household/**/*.jpg", recursive=True) +
                   glob.glob("household/**/*.jpeg", recursive=True) +
                   glob.glob("household/**/*.png", recursive=True))

for path in household_paths:
    test_images.append(path)
    test_labels.append(2)  # non_perishable

test_labels = np.array(test_labels)
print(f"Combined test set: {len(test_images)} images")
print(f"  short_shelf (0):    {(test_labels==0).sum()}")
print(f"  medium_shelf (1):   {(test_labels==1).sum()}")
print(f"  non_perishable (2): {(test_labels==2).sum()}")

---
# Pipeline 1: Shelf-life Classification

## Step 6: Define ImageNet → shelf-life keyword mapping

In [ ]:
# ImageNet label → shelf-life category
IMAGENET_SHORT = [
    "banana", "orange", "strawberry", "apple", "lemon", "pineapple",
    "pomegranate", "fig", "jackfruit", "mango", "broccoli", "cucumber",
    "mushroom", "meat", "egg", "bakery", "bread", "grocery", "fruit",
    "vegetable", "food", "pizza", "hotdog", "pretzel", "bagel", "dough",
    "zucchini", "pepper", "cauliflower", "artichoke", "potato", "cabbage",
    "head cabbage", "corn", "acorn squash", "spaghetti squash",
    "butternut squash", "ice cream", "custard apple"
]

IMAGENET_MEDIUM = [
    "bottle", "can", "jar", "packet", "carton", "sauce", "wine",
    "beer", "juice", "water", "pop", "cup", "coffee", "espresso",
    "milk can", "water bottle", "wine bottle", "beer bottle", "pop bottle"
]

IMAGENET_NON_PERISHABLE = [
    "toilet paper", "paper towel", "soap", "lotion", "shampoo",
    "detergent", "cleaner", "sponge", "brush", "broom", "bucket",
    "basket", "bag", "plastic bag", "trash", "diaper", "tissue",
    "rubber eraser", "pencil", "pen", "notebook", "envelope",
    "container", "box", "crate", "packet", "carton",
    "vacuum", "iron", "washer", "dryer", "lamp", "candle",
    "lighter", "match", "battery", "plug", "switch", "remote",
    "mouse", "keyboard", "monitor", "television", "radio",
    "phone", "calculator", "clock", "watch"
]

def map_to_shelf_life(imagenet_label):
    label = imagenet_label.lower()
    if any(kw in label for kw in IMAGENET_SHORT):
        return 0  # short_shelf
    elif any(kw in label for kw in IMAGENET_MEDIUM):
        return 1  # medium_shelf
    elif any(kw in label for kw in IMAGENET_NON_PERISHABLE):
        return 2  # non_perishable
    else:
        return 2  # default: non_perishable (not food = no expiry concern)

print("3-class keyword mapping ready")

## Step 7: Compare 3 models on Pipeline 1

In [ ]:
from transformers import pipeline as hf_pipeline
import pandas as pd

PIPELINE1_MODELS = {
    "ViT-base": "google/vit-base-patch16-224",
    "ResNet-50": "microsoft/resnet-50",
    "Swin-tiny": "microsoft/swin-tiny-patch4-window7-224",
}

def evaluate_p1(model_key, model_path, test_images, test_labels):
    print(f"\nEvaluating: {model_key}")

    pipe = hf_pipeline("image-classification", model=model_path, device=DEVICE)
    total_params = sum(p.numel() for p in pipe.model.parameters())

    correct = 0
    total = len(test_images)
    inference_times = []

    for i in range(total):
        img = Image.open(test_images[i]).convert("RGB")

        t0 = time.time()
        result = pipe(img, top_k=1)
        inference_times.append(time.time() - t0)

        pred = map_to_shelf_life(result[0]["label"])
        if pred == test_labels[i]:
            correct += 1

    accuracy = correct / total
    avg_ms = np.mean(inference_times) * 1000

    print(f"  Accuracy: {accuracy:.4f} | Speed: {avg_ms:.1f}ms | Params: {total_params/1e6:.1f}M")

    return {
        "Model": model_key,
        "Parameters (M)": f"{total_params/1e6:.1f}M",
        "Accuracy": round(accuracy, 4),
        "Avg Inference (ms)": round(avg_ms, 1),
        "Test Samples": total,
    }

In [ ]:
# Run Pipeline 1 comparison
p1_results = []

for key, path in PIPELINE1_MODELS.items():
    r = evaluate_p1(key, path, test_images, test_labels)
    p1_results.append(r)

df_p1 = pd.DataFrame(p1_results)
print("\n" + "="*60)
print("PIPELINE 1 RESULTS: Shelf-life Classification (3 classes)")
print("="*60)
print(df_p1.to_string(index=False))

## Step 8: Select best model for Pipeline 1

In [ ]:
best_p1 = max(p1_results, key=lambda x: x["Accuracy"])
print(f"Best for Pipeline 1: {best_p1['Model']}")
print(f"  Accuracy: {best_p1['Accuracy']}")
print(f"  Speed: {best_p1['Avg Inference (ms)']}ms")
print(f"\n→ Use this model in app.py: load_shelf_life_classifier()")

---
# Pipeline 3: Image Captioning

## Step 9: Compare 3 captioning models

No accuracy metric for captioning — compare inference speed + output quality.

In [ ]:
PIPELINE3_MODELS = {
    "BLIP-base": "Salesforce/blip-image-captioning-base",
    "BLIP-large": "Salesforce/blip-image-captioning-large",
    "ViT-GPT2": "nlpconnect/vit-gpt2-image-captioning",
}

def evaluate_p3(model_key, model_path, test_images, num_samples=20):
    print(f"\nEvaluating: {model_key}")

    pipe = hf_pipeline("image-to-text", model=model_path, device=DEVICE)
    total_params = sum(p.numel() for p in pipe.model.parameters())

    inference_times = []
    sample_outputs = []

    n = min(num_samples, len(test_images))
    for i in range(n):
        img = Image.open(test_images[i]).convert("RGB")
        t0 = time.time()
        result = pipe(img, max_new_tokens=50)
        inference_times.append(time.time() - t0)
        if i < 5:
            sample_outputs.append(result[0]["generated_text"])

    avg_ms = np.mean(inference_times) * 1000

    print(f"  Speed: {avg_ms:.1f}ms | Params: {total_params/1e6:.1f}M")
    print(f"  Sample outputs:")
    for j, s in enumerate(sample_outputs):
        print(f"    [{j+1}] {s}")

    return {
        "Model": model_key,
        "Parameters (M)": f"{total_params/1e6:.1f}M",
        "Avg Inference (ms)": round(avg_ms, 1),
        "Samples Tested": n,
        "Sample Output": sample_outputs[0] if sample_outputs else "",
    }

In [ ]:
# Run Pipeline 3 comparison
p3_results = []

for key, path in PIPELINE3_MODELS.items():
    r = evaluate_p3(key, path, test_images)
    p3_results.append(r)

df_p3 = pd.DataFrame(p3_results)
print("\n" + "="*60)
print("PIPELINE 3 RESULTS: Image Captioning")
print("="*60)
print(df_p3[["Model", "Parameters (M)", "Avg Inference (ms)"]].to_string(index=False))

## Step 10: Select best model for Pipeline 3

In [ ]:
best_p3 = min(p3_results, key=lambda x: x["Avg Inference (ms)"])
print(f"Fastest for Pipeline 3: {best_p3['Model']} ({best_p3['Avg Inference (ms)']}ms)")
print(f"\nAlso review sample outputs above — pick the one with best quality if speeds are similar.")
print(f"→ Use selected model in app.py: load_image_captioner()")

## Step 11: Export results to Excel

In [ ]:
with pd.ExcelWriter("Pipeline1_3_Experiments.xlsx") as writer:
    df_p1.to_excel(writer, sheet_name="P1 Shelf-life", index=False)
    df_p3[["Model", "Parameters (M)", "Avg Inference (ms)", "Sample Output"]].to_excel(
        writer, sheet_name="P3 Captioning", index=False
    )

print("Saved to Pipeline1_3_Experiments.xlsx")

from google.colab import files
files.download("Pipeline1_3_Experiments.xlsx")
print("Downloaded!")

---
## Notebook Summary

| Step | Content |
|------|---------|
| 1-3 | Setup: dependencies, GPU check, Kaggle credentials |
| 4-5 | Download Grocery Store Dataset + Household Products, build 3-class test set |
| 6-8 | **Pipeline 1:** ViT vs ResNet vs Swin + keyword mapping → 3-class accuracy + speed → select best |
| 9-10 | **Pipeline 3:** BLIP-base vs BLIP-large vs ViT-GPT2 → speed + quality → select best |
| 11 | Export Excel + auto-download |

**Datasets:**
- Grocery Store Dataset: https://github.com/marcusklasson/GroceryStoreDataset
- Kaggle Household Products: https://www.kaggle.com/datasets/taru149/householdproducts